# 03 · Experiments: DeepLabV3+ variants and backbones

Notebook 02 compared three architectures (baseline CNN, U-Net, DeepLabV3+ with ResNet50). Observations there
led to two questions this notebook answers with a controlled experiment:

1. **Backbone:** does a deeper (ResNet101, ResNet152) or a more efficient (Xception) pretrained encoder help?
2. **Encoder design:** does the **atrous encoder** (AE: full backbone, last stage dilated instead of strided)
   beat the cut-off encoder of the standard variant used here?

| | ResNet101 | ResNet152 | Xception |
|---|---|---|---|
| **DeepLabV3+** (backbone cut after the 1/16 stage) | 36.9 M | 52.6 M | 24.6 M |
| **AE-DeepLabV3+** (full backbone, atrous last stage, output stride 16) | 59.5 M | 75.2 M | 37.7 M |

**Controlled comparison:** all six use the same data, splits, augmentation, loss, optimiser, learning rate
(3e-4), batch size (8, so the largest model fits on the shared GPU), and stopping rules as notebook 02. Only the
model changes. Each experiment is recorded as *hypothesis → result → lesson*.

Runs are resumable: a finished experiment is loaded instead of retrained, so the notebook can be restarted after
an interruption.

In [ ]:
import os, shutil, subprocess
if shutil.which("nvidia-smi"):
    free = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True).stdout.split()
    os.environ["CUDA_VISIBLE_DEVICES"] = str(max(range(len(free)), key=lambda i: int(free[i])))
    print("Free GPU memory (MB):", free, "-> using GPU", os.environ["CUDA_VISIBLE_DEVICES"])
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
keras.mixed_precision.set_global_policy("mixed_float16")
keras.utils.set_random_seed(42)

from src import config as C, model as M, patching as P, training as T

print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
SMOKE_TEST = False     # True: 64 random patches per split, 2 epochs - only checks that the code runs

BATCH = 8
LR = 3e-4
MAX_EPOCHS = 2 if SMOKE_TEST else 50
PATIENCE = 8

WEIGHTS_DIR = Path(os.environ.get("LULC_WEIGHTS_DIR", ROOT / "weights"))
OUT_DIR = WEIGHTS_DIR / ("experiments_smoke_test" if SMOKE_TEST else "experiments")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stats = M.load_stats()
X_train, y_train, _ = T.load_split("train", smoke=SMOKE_TEST)
X_val, y_val, _ = T.load_split("val", smoke=SMOKE_TEST)
X_test, y_test, meta_test = T.load_split("test", in_memory=False, smoke=SMOKE_TEST)
class_weights = M.median_frequency_weights(P.class_distribution("train").pixels)
data = dict(X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, stats=stats, class_weights=class_weights)
print(f"train {len(X_train)} | val {len(X_val)} | test {len(X_test)} patches")

## Reference: notebook 02

The three models of notebook 02 (batch 16), for comparison.

In [ ]:
reference = {}
results_02 = WEIGHTS_DIR / "results.json"
if results_02.exists():
    r02 = json.loads(results_02.read_text())
    for n, t in r02["training"].items():
        reference[n] = {"params (M)": t["params"] / 1e6, "train mIoU": t["train_mIoU"], "val mIoU": t["val_mIoU"],
                        "gap": t["gap"], "test OA": r02["test"][n]["OA"], "test mIoU": r02["test"][n]["mIoU"],
                        "test mF1": r02["test"][n]["mF1"]}
    display(pd.DataFrame(reference).T.round(4))
else:
    print("Run notebook 02 first to see its results here.")

In [ ]:
runs = {}


def run(name, title):
    model, history, info = T.train_model(name, data, OUT_DIR, lr=LR, batch=BATCH, max_epochs=MAX_EPOCHS,
                                         patience=PATIENCE)
    T.plot_history(history, title)
    test = T.evaluate(model, X_test, y_test, stats)
    runs[name] = dict(title=title, history=history, info=info, test=test)
    del model
    keras.backend.clear_session()
    return pd.Series({**{k: v for k, v in info.items() if k != "verdict"}, "test OA": test["OA"],
                      "test mIoU": test["mIoU"], "test mF1": test["mF1"], "verdict": info["verdict"]})

## E1 · DeepLabV3+ · Xception

Xception replaces ordinary convolutions with depthwise separable ones: far fewer parameters (24.6 M) for a similar receptive field. **Hypothesis:** with only 26 training regions, a leaner encoder overfits less and generalises at least as well as ResNet.

In [ ]:
run("deeplabv3plus_xception", "E1 · DeepLabV3+ · Xception")

## E2 · DeepLabV3+ · ResNet101

Twice as many residual blocks in stage 4 as the ResNet50 model of notebook 02 (36.9 M parameters). **Hypothesis:** deeper features separate classes that need more context (crops vs. rangeland, wetlands).

In [ ]:
run("deeplabv3plus_resnet101", "E2 · DeepLabV3+ · ResNet101")

## E3 · DeepLabV3+ · ResNet152

The deepest backbone (52.6 M parameters). **Hypothesis:** returns diminish; more capacity mainly fits the training regions better (larger train–validation gap).

In [ ]:
run("deeplabv3plus_resnet152", "E3 · DeepLabV3+ · ResNet152")

## E4 · AE-DeepLabV3+ · Xception

Atrous encoder: the full Xception including the exit flow, with dilated instead of strided convolutions, so features stay at 1/16 resolution (37.7 M parameters). **Hypothesis:** the larger receptive field helps large homogeneous classes.

In [ ]:
run("ae_deeplabv3plus_xception", "E4 · AE-DeepLabV3+ · Xception")

## E5 · AE-DeepLabV3+ · ResNet101

Full ResNet101 with a dilated stage 5 (59.5 M parameters). Compared with the cut-off DeepLabV3+ · ResNet101, this isolates the effect of the atrous encoder.

In [ ]:
run("ae_deeplabv3plus_resnet101", "E5 · AE-DeepLabV3+ · ResNet101")

## E6 · AE-DeepLabV3+ · ResNet152

The largest model (75.2 M parameters). **Hypothesis:** the most prone to overfitting; only worth it if the validation mIoU clearly improves.

In [ ]:
run("ae_deeplabv3plus_resnet152", "E6 · AE-DeepLabV3+ · ResNet152")

## Comparison

In [ ]:
rows = {}
for n, r in runs.items():
    i, t = r["info"], r["test"]
    rows[r["title"]] = {"params (M)": i["params"] / 1e6, "epochs": i["epochs_run"], "best epoch": i["best_epoch"],
                        "minutes": i["minutes"], "train mIoU": i["train_mIoU"], "val mIoU": i["val_mIoU"],
                        "gap": i["gap"], "test OA": t["OA"], "test mIoU": t["mIoU"], "test mF1": t["mF1"],
                        "verdict": i["verdict"]}
comparison = pd.DataFrame(rows).T
comparison

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
markers = {"resnet101": "o", "resnet152": "s", "xception": "^"}
for n, r in runs.items():
    backbone = n.split("_")[-1]
    color = "#dc2626" if n.startswith("ae_") else "#2563eb"
    ax[0].scatter(r["info"]["params"] / 1e6, r["info"]["val_mIoU"], marker=markers[backbone], color=color, s=90)
    ax[0].annotate(r["title"].split("· ")[1], (r["info"]["params"] / 1e6, r["info"]["val_mIoU"]),
                   textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax[1].plot(r["history"].val_miou, color=color, ls="-" if backbone != "xception" else "--",
               marker=markers[backbone], markevery=5, label=r["title"])
for n, ref in reference.items():
    if n == "deeplabv3plus":
        ax[0].scatter(ref["params (M)"], ref["val mIoU"], marker="D", color="#6b7280", s=90)
        ax[0].annotate("ResNet50 (nb 02)", (ref["params (M)"], ref["val mIoU"]), textcoords="offset points",
                       xytext=(6, 4), fontsize=8)
ax[0].set_xlabel("parameters (millions)"); ax[0].set_ylabel("best validation mIoU")
ax[0].set_title("Capacity vs. validation mIoU (blue: DeepLabV3+, red: AE-DeepLabV3+)")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("validation mIoU"); ax[1].set_title("Validation mIoU per epoch")
ax[1].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "experiments_comparison.png", dpi=150)
plt.show()

### Effect of the atrous encoder (AE minus standard, same backbone)

In [ ]:
effect = {}
for b in M.BACKBONES:
    std, ae = runs.get(f"deeplabv3plus_{b}"), runs.get(f"ae_deeplabv3plus_{b}")
    if std and ae:
        effect[b] = {"Δ val mIoU": ae["info"]["val_mIoU"] - std["info"]["val_mIoU"],
                     "Δ test mIoU": ae["test"]["mIoU"] - std["test"]["mIoU"],
                     "Δ gap": ae["info"]["gap"] - std["info"]["gap"],
                     "Δ params (M)": (ae["info"]["params"] - std["info"]["params"]) / 1e6}
pd.DataFrame(effect).T.round(4)

### Per-class test IoU

In [ ]:
per_class = pd.DataFrame({r["title"]: r["test"]["per_class"]["IoU"] for r in runs.values()})
fig, ax = plt.subplots(figsize=(12, 4.5))
im = ax.imshow(per_class.values, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_yticks(range(len(per_class)), per_class.index)
ax.set_xticks(range(per_class.shape[1]), [c.replace(" · ", "\n") for c in per_class.columns], fontsize=8)
for i in range(per_class.shape[0]):
    for j in range(per_class.shape[1]):
        v = per_class.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color="white" if v < 0.5 else "black")
plt.colorbar(im, fraction=0.03)
ax.set_title("Test IoU per class")
plt.tight_layout()
plt.savefig(OUT_DIR / "experiments_per_class_iou.png", dpi=150)
plt.show()

## Findings

*(Written after the run, one line per experiment: was the hypothesis confirmed, and what does it teach?)*

| Experiment | Hypothesis confirmed? | Lesson |
|---|---|---|
| E1 · Xception | | |
| E2 · ResNet101 | | |
| E3 · ResNet152 | | |
| E4 · AE Xception | | |
| E5 · AE ResNet101 | | |
| E6 · AE ResNet152 | | |

**Selection:** the experiment with the highest *validation* mIoU is saved as
`weights/experiments/best_experiment.keras`. It replaces the notebook-02 model in the prototype only if its
validation mIoU is higher.

In [ ]:
summary = {n: {"title": r["title"], **{k: v for k, v in r["info"].items()},
                "test": {"OA": float(r["test"]["OA"]), "mIoU": float(r["test"]["mIoU"]), "mF1": float(r["test"]["mF1"]),
                         "per_class_IoU": r["test"]["per_class"]["IoU"].round(4).to_dict(),
                         "per_class_F1": r["test"]["per_class"]["F1"].round(4).to_dict()}}
           for n, r in runs.items()}
(OUT_DIR / "experiments_results.json").write_text(json.dumps(summary, indent=2, default=float))

best = max(runs, key=lambda n: runs[n]["info"]["val_mIoU"])
shutil.copy(OUT_DIR / f"{best}.keras", OUT_DIR / "best_experiment.keras")
ref_val = reference.get(json.loads(results_02.read_text())["selected_model"], {}).get("val mIoU") if results_02.exists() else None
print(f"Best experiment on validation mIoU: {runs[best]['title']} ({runs[best]['info']['val_mIoU']:.4f})")
print(f"Notebook-02 selected model: {ref_val}")
if ref_val is not None and runs[best]["info"]["val_mIoU"] > ref_val:
    print("-> better than notebook 02: use weights/experiments/best_experiment.keras in the prototype")
else:
    print("-> not better than notebook 02: the prototype keeps weights/best_model.keras")